# FastGen Final Workflow

End-to-end notebook for training and evaluating FastGen student models on CIFAR-10.

**This notebook covers:**
1. **Setting up the environment** — Install FastGen, configure conda, and verify GPU availability
2. **Downloading data and models** — Obtain CIFAR-10 dataset, FID reference statistics, and configure W&B credentials
3. **Training the student model** — Train a DMD2 student model using knowledge distillation from a teacher EDM model
4. **Evaluating both models** — Compare teacher (EDM) and student (DMD2) model performance using FID scores

## Part 1: Setting Up the Environment

Set up a clean conda environment with FastGen and required dependencies.

### 1.1 Verify Python, pip, and conda availability

Run this first to confirm the notebook environment already has Python, pip, and conda. If conda is missing, use the optional Miniconda install block below.

In [47]:
!which python3 && which pip && conda --version 2>&1 || echo "conda not found"

/opt/venv/bin/python3
/usr/local/bin/pip
/bin/bash: line 1: conda: command not found
conda not found


### 1.2 Install Miniconda (only if conda is not already installed)

If `conda` is missing, uncomment and run this block. This is optional on systems where conda is already installed.

In [ ]:
# Optional: install Miniconda if conda is not available
# curl -sL https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -o /tmp/miniconda.sh
# bash /tmp/miniconda.sh -b -p $HOME/miniconda3
# bash -lc "source $HOME/.bashrc && conda --version"

### 1.3 Create the `fastgen` conda environment

This cell removes any existing `fastgen` environment and recreates it cleanly with Python 3.12.3.

In [ ]:
source ~/miniconda3/etc/profile.d/conda.sh
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true
conda env remove -n fastgen -y 2>/dev/null || true
conda create -y -n fastgen python=3.12.3 pip

### 1.4 Activate the environment and verify Python

Each shell block that uses `conda activate` must source conda's hook first. This cell confirms the environment is ready.

In [ ]:
source ~/miniconda3/etc/profile.d/conda.sh
conda activate fastgen
python --version

### 1.5 Install FastGen and register the Jupyter kernel

Install FastGen in editable mode and register the conda environment as a Jupyter kernel so Python cells execute in the correct environment.

In [ ]:
source ~/miniconda3/etc/profile.d/conda.sh
conda activate fastgen
if [ -d .git ]; then
  echo "Installing from current FastGen repository"
  pip install -e .
else
  git clone https://github.com/NVlabs/FastGen.git repo_clone
  cd repo_clone
  pip install -e .
fi
pip install pandas
python -m ipykernel install --user --name fastgen --display-name "Python (fastgen GPU)"

### 1.6 Verify the installation

Import FastGen and PyTorch to confirm the environment is correctly installed and verify GPU availability.

In [ ]:
source ~/miniconda3/etc/profile.d/conda.sh
conda activate fastgen
python - <<'PY'
import torch
ok = True
try:
    import fastgen
except Exception as e:
    print('FAILED import fastgen:', e)
    ok = False
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
print('RESULT:', 'PASS' if (ok and torch.cuda.is_available()) else 'FAIL')
PY

## Part 2: Downloading Data, Models, and Adding Credentials

Prepare all required assets for training and evaluation.

### 2.1 Download CIFAR-10 Dataset

In [41]:
%%bash
source ~/miniconda3/etc/profile.d/conda.sh
conda activate fastgen
python scripts/download_data.py --dataset cifar10


bash: line 1: /root/miniconda3/etc/profile.d/conda.sh: No such file or directory
bash: line 2: conda: command not found


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Multiple distributions found for package modelopt. Picked distribution: nvidia-modelopt
Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (64)xFormers not available


[Jun 23, 2026 - 10:15:07 | INFO | __main__:main:623 ] FastGen Dataset Setup: cifar10
[Jun 23, 2026 - 10:15:07 | INFO | __main__:main:624 ] Data directory:       /workspace/user_homes/hseth/test/FastGen/FASTGEN_OUTPUT/DATA
[Jun 23, 2026 - 10:15:07 | INFO | __main__:main:625 ] Checkpoint directory: /workspace/user_homes/hseth/test/FastGen/FASTGEN_OUTPUT/MODEL
[Jun 23, 2026 - 10:15:07 | INFO | __main__:clone_repo:195 ] Cloning edm repo to /tmp/tmpt7rv7b7b/edm...
[Jun 23, 2026 - 10:15:09 | INFO | __main__:main:643 ] ==================================================
[Jun 23, 2026 - 10:15:09 | INFO | __main__:main:644 ] Processing CIFAR-10
[Jun 23, 2026 - 10:15:09 | INFO | __main__:main:645 ] ==================================================
[Jun 23, 2026 - 10:15:09 | INFO | __main__:run_dataset_tool:221 ] Dataset already exists: FASTGEN_OUTPUT/DATA/cifar10/cifar10-32x32.zip
[Jun 23, 2026 - 10:15:09 | INFO | __main__:prepare_models:340 ] All EDM CIFAR-10 models already exist:
[Jun 23, 2026

### 2.2 Download CIFAR-10 FID Reference Statistics

Download pre-computed FID reference statistics for CIFAR-10 to enable FID score computation during evaluation.

In [ ]:
source ~/miniconda3/etc/profile.d/conda.sh
conda activate fastgen
python scripts/download_data.py --dataset cifar10 --compute-fid-refs

### 2.3 Configure W&B (Weights & Biases) Token

Set up W&B credentials for experiment tracking and logging during training.

In [ ]:
import os
token = #'<YOUR_WANDB_API_KEY>'  # Replace with your actual W&B API key
os.makedirs('credentials', exist_ok=True)
with open('credentials/wandb_api.txt', 'w', encoding='utf-8') as f:
    f.write(token)
os.environ['WANDB_API_KEY'] = token
print('W&B token saved to credentials/wandb_api.txt')

W&B token saved to credentials/wandb_api.txt


## Part 3: Training the Student Model

### Configuration Details

**Current Config:** `config_dmd2_test.py`

- **Model Type:** DMD2 (Distilled Model Distillation 2) — a knowledge distillation approach
- **Teacher Model:** Pre-trained EDM (Elucidating the Design Space of Diffusion-Based Generative Models) on CIFAR-10
- **Student Model:** Lightweight fast sampler trained via adversarial distillation
- **Distillation Method:** Uses discriminator-guided training to match teacher distributions with fewer sampling steps

**Other Available Configs for CIFAR-10 distillation:**
- `config_cm_cifar10.py` — Consistency Model distillation
- `config_tcm_cifar10.py` — Trajectory Consistency Model
- `config_scd_cifar10.py` — Score-based Consistency Distillation
- `config_sct_cifar10.py` — Sequence Consistency Training
- `config_mf_cifar10.py` — Matching Forward process
- `config_cm_cifar10_fast.py` — Fast Consistency Model variant

### 3.1 Train the Student Model

*Note*: In this workshop we are only running the test config config_dmd2_test.py and training the student model for about 5k steps. Full convergence of the DMD2 student model takes roughly 100k steps and about 12-14 hours on 8 x H100. Because we are not training to full convergence here, the evaluation results will also reflect this partial training run

In [43]:
import os
CONFIG = 'fastgen/configs/experiments/EDM/config_dmd2_test.py'
LOG_NAME = 'student_run'
NUM_GPUS = 4
os.environ['FASTGEN_OUTPUT_ROOT'] = os.getenv('FASTGEN_OUTPUT_ROOT', 'FASTGEN_OUTPUT')
print(f'Training student model with log_config.name={LOG_NAME}')
!torchrun --nproc_per_node={NUM_GPUS} train.py --config={CONFIG} - trainer.ddp=True log_config.name={LOG_NAME}

Training student model with log_config.name=student_run


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr/local/lib/python3.12/dist-packages/torch/cud

## Part 4: Evaluating Both Models

Compare teacher (original EDM) and student (distilled DMD2) model performance on CIFAR-10.

### 4.1 Evaluate Teacher Model (Original EDM)

**Config Used:** `config_sft_edm_cifar10.py`

This evaluation uses the original pre-trained EDM model without any distillation as our teacher baseline. We use `config_sft_edm_cifar10.py` (SFT = Supervised Fine-Tuning) which employs the standard 18 sampling steps typical for EDM models. This generates high-quality samples that serve as our reference for comparing the distilled student model's performance.

We copy the baseline teacher checkpoint from its original repository path into the FastGen output checkpoint directory because the FastGen evaluation scripts expect a checkpoint in the local `FASTGEN_OUTPUT_ROOT/.../checkpoints/` structure and in the wrapped FastGen format `({"model": {"net": raw}, ...})`. This makes the pre-trained teacher model appear like any other FastGen-trained checkpoint so the FID evaluation can load it cleanly using the same pipeline.

> ⚠️ **Note:** If running evaluation again, delete the old evaluation folder first: `rm -rf $FASTGEN_OUTPUT_ROOT/fastgen/edm_cifar10_sft/EDM_original/`

In [55]:
import os, json, torch, time

# ── Edit these ───────────────────────────────────────────────────────
CKPT_ROOT_DIR       = os.environ.get("CKPT_ROOT_DIR", "FASTGEN_OUTPUT/MODEL")
FASTGEN_OUTPUT_ROOT = os.environ.get("FASTGEN_OUTPUT_ROOT", "FASTGEN_OUTPUT")
LOG_GROUP           = "edm_cifar10_sft"
LOG_NAME            = "EDM_original"
TEACHER_EVAL_SAMPLES = 500
# ─────────────────────────────────────────────────────────────────────

src_path = f"{CKPT_ROOT_DIR}/cifar10/edm-cifar10-32x32-cond-vp.pth"
dst_dir  = f"{FASTGEN_OUTPUT_ROOT}/fastgen/{LOG_GROUP}/{LOG_NAME}/checkpoints"
dst_path = f"{dst_dir}/0000001.pth"

# 1. Load raw checkpoint
assert os.path.exists(src_path), f"Checkpoint not found: {src_path}"
raw = torch.load(src_path, map_location="cpu")
print(f"✅ Loaded raw checkpoint from: {src_path}")

# 2. Wrap and save
os.makedirs(dst_dir, exist_ok=True)
torch.save({"model": {"net": raw}, "iteration": 1}, dst_path)
print(f"✅ Wrapped checkpoint saved to: {dst_path}")

# 3. Run FID evaluation
start_time = time.time()
return_code = os.system(
    f"python scripts/fid/compute_fid_from_ckpts.py --config fastgen/configs/experiments/EDM/config_sft_edm_cifar10.py - eval.num_samples={TEACHER_EVAL_SAMPLES} log_config.name={LOG_NAME} log_config.group={LOG_GROUP}"
)
teacher_eval_time = time.time() - start_time
print(f"\nTeacher evaluation finished in {teacher_eval_time:.2f} seconds")
if return_code != 0:
    raise RuntimeError(f"Teacher evaluation failed with exit code {return_code}")

# 4. Print FID result
fid_path = f"{FASTGEN_OUTPUT_ROOT}/fastgen/{LOG_GROUP}/{LOG_NAME}/samples/fid.json"
with open(fid_path) as f:
    result = json.load(f)
print("\n📊 FID Results:")
for ckpt, fid in zip(result["ckpt_num"], result["fid"]):
    print(f"   iter {ckpt:>7d} → FID = {fid:.4f}")


✅ Loaded raw checkpoint from: FASTGEN_OUTPUT/MODEL/cifar10/edm-cifar10-32x32-cond-vp.pth


✅ Wrapped checkpoint saved to: FASTGEN_OUTPUT/fastgen/edm_cifar10_sft/EDM_original/checkpoints/0000001.pth


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (64)Multiple distributions found for package modelopt. Picked distribution: nvidia-modelopt
xFormers not available


[Jun 23, 2026 - 10:43:24 | INFO | fastgen.configs.config_utils:serialize_config:315 ] Config is saved at FASTGEN_OUTPUT/fastgen/edm_cifar10_sft/EDM_original/samples/config.yaml
[Jun 23, 2026 - 10:43:24 | INFO | fastgen.utils.scripts:setup:87 ] No DDP or FSDP parallelism
[Jun 23, 2026 - 10:43:24 | INFO | fastgen.utils.scripts:setup:113 ] Changing gradient accumulation rounds from 1 to 64 to match requested global batch size.
[Jun 23, 2026 - 10:43:24 | CRITICAL | fastgen.utils.scripts:setup:118 ] Global batch size: 16384 (Batch size per GPU: 256, Gradient accumulation rounds: 64, World size: 1)
[Jun 23, 2026 - 10:43:24 | CRITICAL | fastgen.utils.scripts:set_cuda_backend:46 ] cuDNN deterministic: False, cuDNN benchmark: True, enable TF32: True
[Jun 23, 2026 - 10:43:24 | INFO | fastgen.utils.basic_utils:set_random_seed:144 ] Using random seed 0.
[Jun 23, 2026 - 10:43:24 | CRITICAL | fastgen.methods.model:set_precision:172 ] Model and data precision: torch.float32. AMP training precision: N

100%|██████████| 2/2 [01:18<00:00, 39.15s/batch]


[Jun 23, 2026 - 10:44:54 | INFO | scripts.fid.fid:calc:114 ] Loading dataset reference statistics from "FASTGEN_OUTPUT/DATA/fid-refs/cifar10-32x32.npz"...
[Jun 23, 2026 - 10:44:54 | INFO | scripts.fid.fid:calc:137 ] Loading Inception-v3 model...
[Jun 23, 2026 - 10:44:55 | INFO | scripts.fid.fid:calculate_inception_stats:42 ] Loading images from "FASTGEN_OUTPUT/fastgen/edm_cifar10_sft/EDM_original/samples/iter_1"...
[Jun 23, 2026 - 10:44:55 | INFO | scripts.fid.fid:calculate_inception_stats:62 ] Calculating statistics for 500 images...


100%|██████████| 2/2 [00:03<00:00,  1.72s/batch]


[Jun 23, 2026 - 10:44:58 | INFO | scripts.fid.fid:calc:158 ] Calculating FID for FASTGEN_OUTPUT/fastgen/edm_cifar10_sft/EDM_original/samples/iter_1... 
[Jun 23, 2026 - 10:45:06 | INFO | scripts.fid.fid:calc:161 ] path: FASTGEN_OUTPUT/fastgen/edm_cifar10_sft/EDM_original/samples/iter_1
[Jun 23, 2026 - 10:45:06 | INFO | scripts.fid.fid:calc:162 ] FID: 56.478468215581735
[Jun 23, 2026 - 10:45:06 | INFO | scripts.fid.fid:calc:163 ] ====================
[Jun 23, 2026 - 10:45:07 | INFO | __main__:remove_iter_dirs:63 ] ✓ Removed 1 directorie(s) (errors: 0) from /workspace/user_homes/hseth/test/FastGen/FASTGEN_OUTPUT/fastgen/edm_cifar10_sft/EDM_original/samples

Teacher evaluation finished in 117.04 seconds

📊 FID Results:
   iter       1 → FID = 56.4785


### 4.2 Evaluate Student Model

**Config Used:** `config_dmd2_cifar10.py`

This evaluation uses the distilled student model trained in Part 3, using `config_dmd2_cifar10.py` (DMD2 = Distilled Model Distillation 2) for evaluation. The student model is configured with just 1 sampling step (or as configured in the distilled model), allowing us to evaluate the efficiency and quality trade-off compared to the teacher. The copied checkpoint is automatically retrieved from the training output directory, ensuring we evaluate the trained model with consistency.

> ⚠️ **Note:** If running evaluation again, delete the old evaluation folder first: `rm -rf $FASTGEN_OUTPUT_ROOT/fastgen/evaluation/student_run/`

In [50]:
import os, json, time, shutil, glob

# ── Edit these ───────────────────────────────────────────────────────
CKPT_ROOT_DIR       = os.environ.get("CKPT_ROOT_DIR", "FASTGEN_OUTPUT/MODEL")
FASTGEN_OUTPUT_ROOT = os.environ.get("FASTGEN_OUTPUT_ROOT", "FASTGEN_OUTPUT")
LOG_GROUP           = "cifar10"
LOG_NAME            = "student_run"
STUDENT_EVAL_SAMPLES = 500
# ─────────────────────────────────────────────────────────────────────

# Step 1: Find the last checkpoint from training
train_ckpt_dir = f"{FASTGEN_OUTPUT_ROOT}/fastgen/{LOG_GROUP}/{LOG_NAME}/checkpoints"
ckpt_files = sorted(glob.glob(os.path.join(train_ckpt_dir, "*.pth")))
assert ckpt_files, f"No checkpoints found in: {train_ckpt_dir}"

last_ckpt = ckpt_files[-1]
print(f"✅ Found last checkpoint: {last_ckpt}")

# Step 2: Copy to evaluation directory with same log_name structure
eval_ckpt_dir = f"{FASTGEN_OUTPUT_ROOT}/fastgen/evaluation/{LOG_NAME}/checkpoints"
os.makedirs(eval_ckpt_dir, exist_ok=True)

eval_ckpt_path = os.path.join(eval_ckpt_dir, os.path.basename(last_ckpt))
shutil.copy2(last_ckpt, eval_ckpt_path)
print(f"✅ Copied checkpoint to: {eval_ckpt_path}")

# Step 3: Run FID evaluation on the copied checkpoint
print(f"\nEvaluating student model: {LOG_NAME}")
start_time = time.time()
return_code = os.system(
    f"python scripts/fid/compute_fid_from_ckpts.py --config fastgen/configs/experiments/EDM/config_dmd2_cifar10.py - eval.num_samples={STUDENT_EVAL_SAMPLES} log_config.name={LOG_NAME} log_config.group=evaluation"
)
student_eval_time = time.time() - start_time
print(f"\nStudent evaluation finished in {student_eval_time:.2f} seconds")
if return_code != 0:
    raise RuntimeError(f"Student evaluation failed with exit code {return_code}")

# Step 4: Print FID results
fid_path = f"{FASTGEN_OUTPUT_ROOT}/fastgen/evaluation/{LOG_NAME}/samples/fid.json"
with open(fid_path) as f:
    result = json.load(f)
print("\n📊 FID Results:")
for ckpt, fid in zip(result["ckpt_num"], result["fid"]):
    print(f"   iter {ckpt:>7d} → FID = {fid:.4f}")

✅ Found last checkpoint: FASTGEN_OUTPUT/fastgen/cifar10/student_run/checkpoints/0005000.pth


✅ Copied checkpoint to: FASTGEN_OUTPUT/fastgen/evaluation/student_run/checkpoints/0005000.pth

Evaluating student model: student_run


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (64)Multiple distributions found for package modelopt. Picked distribution: nvidia-modelopt
xFormers not available


[Jun 23, 2026 - 10:23:29 | INFO | fastgen.configs.config_utils:serialize_config:315 ] Config is saved at FASTGEN_OUTPUT/fastgen/evaluation/student_run/samples/config.yaml
[Jun 23, 2026 - 10:23:29 | INFO | fastgen.utils.scripts:setup:87 ] No DDP or FSDP parallelism
[Jun 23, 2026 - 10:23:29 | INFO | fastgen.utils.scripts:setup:113 ] Changing gradient accumulation rounds from 1 to 8 to match requested global batch size.
[Jun 23, 2026 - 10:23:29 | CRITICAL | fastgen.utils.scripts:setup:118 ] Global batch size: 2048 (Batch size per GPU: 256, Gradient accumulation rounds: 8, World size: 1)
[Jun 23, 2026 - 10:23:29 | CRITICAL | fastgen.utils.scripts:set_cuda_backend:46 ] cuDNN deterministic: False, cuDNN benchmark: True, enable TF32: True
[Jun 23, 2026 - 10:23:29 | INFO | fastgen.utils.basic_utils:set_random_seed:144 ] Using random seed 0.
[Jun 23, 2026 - 10:23:29 | CRITICAL | fastgen.methods.model:set_precision:172 ] Model and data precision: torch.float32. AMP training precision: None. AMP 

### 4.3 Performance Comparison

Compare evaluation timing and speedup between teacher and student models.

In [52]:
try:
    print("⏱️  Evaluation Timing Comparison:")
    print(f"   Teacher evaluation: {teacher_eval_time:.2f} seconds")
    print(f"   Student evaluation: {student_eval_time:.2f} seconds")
    speedup = teacher_eval_time / student_eval_time if student_eval_time > 0 else float('inf')
    print(f"   Speedup ratio (Teacher/Student): {speedup:.2f}x")
except NameError as e:
    print('Timing variables not found. Make sure both evaluation cells have been run.')

⏱️  Evaluation Timing Comparison:
   Teacher evaluation: 117.22 seconds
   Student evaluation: 26.74 seconds
   Speedup ratio (Teacher/Student): 4.38x


## Documentation

Detailed documentation is available in each component's README:

| Component | Documentation | Description |
|-----------|---------------|-------------|
| **Methods** | [fastgen/methods/README.md](fastgen/methods/README.md) | Training methods (sCM, MeanFlow, DMD2, Self-Forcing, etc.) |
| **Networks** | [fastgen/networks/README.md](fastgen/networks/README.md) | Network architectures (EDM, SD, SDXL, Flux, Qwen-Image, WAN, CogVideoX, Cosmos) and pretrained models |
| **Configs** | [fastgen/configs/README.md](fastgen/configs/README.md) | Configuration system, environment variables, and creating custom configs |
| **Datasets** | [fastgen/datasets/README.md](fastgen/datasets/README.md) | Dataset preparation and WebDataset loaders |
| **Callbacks** | [fastgen/callbacks/README.md](fastgen/callbacks/README.md) | Training callbacks (EMA, logging, gradient clipping, etc.) |
| **Inference** | [scripts/README.md](scripts/README.md) | Inference modes (T2I, T2V, I2V, V2V, etc.) and FID evaluation |
| **Third Party** | [fastgen/third_party/README.md](fastgen/third_party/README.md) | Third-party dependencies (Depth Anything V2, etc.) |